In [0]:
import json
import time
import uuid
import requests

from datetime import datetime
from delta.tables import DeltaTable

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    ArrayType
)

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    arrays_zip,
    explode,
    to_timestamp
)



BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]



INCREMENTAL_START_YEAR = 2021
INCREMENTAL_END_YEAR = 2025

LOAD_TYPE = "INCREMENTAL"

RAW_INCREMENTAL_PATH = (
    "/Volumes/workspace/default/products/"
    "european_climate/raw/incremental"
)

BRONZE_TABLE = "bronze_weather"
SILVER_TABLE = "silver_weather"

EXECUTION_LOG_TABLE = "pipeline_execution_logs"
ERROR_LOG_TABLE = "pipeline_error_logs"

NORMAL_DELAY_SECONDS = 6.0
MAX_RETRIES = 8


FORCE_REFRESH_RAW = True

CITIES = [
    {
        "city": "Oslo",
        "country": "Norway",
        "region": "Northern Europe",
        "latitude": 59.9139,
        "longitude": 10.7522
    },

    {
        "city": "Stockholm",
        "country": "Sweden",
        "region": "Northern Europe",
        "latitude": 59.3293,
        "longitude": 18.0686
    },

    {
        "city": "Helsinki",
        "country": "Finland",
        "region": "Northern Europe",
        "latitude": 60.1699,
        "longitude": 24.9384
    },

    {
        "city": "Copenhagen",
        "country": "Denmark",
        "region": "Northern Europe",
        "latitude": 55.6761,
        "longitude": 12.5683
    },

    {
        "city": "London",
        "country": "United Kingdom",
        "region": "Western Europe",
        "latitude": 51.5074,
        "longitude": -0.1278
    },

    {
        "city": "Berlin",
        "country": "Germany",
        "region": "Central Europe",
        "latitude": 52.5200,
        "longitude": 13.4050
    },

    {
        "city": "Madrid",
        "country": "Spain",
        "region": "Southern Europe",
        "latitude": 40.4168,
        "longitude": -3.7038
    },

    {
        "city": "Rome",
        "country": "Italy",
        "region": "Southern Europe",
        "latitude": 41.9028,
        "longitude": 12.4964
    },

    {
        "city": "Warsaw",
        "country": "Poland",
        "region": "Eastern Europe",
        "latitude": 52.2297,
        "longitude": 21.0122
    },
]

TOTAL_BATCHES = (
    len(CITIES)
    * (INCREMENTAL_END_YEAR - INCREMENTAL_START_YEAR + 1)
)

print("===== INCREMENTAL CONFIGURATION =====")
print("Cities:", len(CITIES))
print(
    "Years:",
    INCREMENTAL_START_YEAR,
    "→",
    INCREMENTAL_END_YEAR
)
print("Batches:", TOTAL_BATCHES)
print("Load type:", LOAD_TYPE)
print("Fresh API fetch:", FORCE_REFRESH_RAW)

===== INCREMENTAL CONFIGURATION =====
Cities: 9
Years: 2021 → 2025
Batches: 45
Load type: INCREMENTAL
Fresh API fetch: True


In [0]:
hourly_units_schema = StructType([
    StructField("time", StringType(), True),

    *[
        StructField(name, StringType(), True)
        for name in WEATHER_COLUMNS
    ],
])

hourly_schema = StructType([
    StructField("time", ArrayType(StringType()), True),

    StructField(
        "temperature_2m",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "relative_humidity_2m",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "apparent_temperature",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "precipitation",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "rain",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "snowfall",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "weather_code",
        ArrayType(IntegerType()),
        True
    ),

    StructField(
        "pressure_msl",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "surface_pressure",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "cloud_cover",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "wind_speed_10m",
        ArrayType(DoubleType()),
        True
    ),

    StructField(
        "wind_direction_10m",
        ArrayType(DoubleType()),
        True
    ),
])

raw_weather_schema = StructType([
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),

    StructField(
        "generationtime_ms",
        DoubleType(),
        True
    ),

    StructField(
        "utc_offset_seconds",
        IntegerType(),
        True
    ),

    StructField("timezone", StringType(), True),

    StructField(
        "timezone_abbreviation",
        StringType(),
        True
    ),

    StructField("elevation", DoubleType(), True),

    StructField(
        "hourly_units",
        hourly_units_schema,
        True
    ),

    StructField(
        "hourly",
        hourly_schema,
        True
    ),
])


def make_batch_id(city_info, start_date, end_date):
    safe_city = (
        city_info["city"]
        .lower()
        .replace(" ", "_")
    )

  
    return (
        f"incremental_"
        f"{safe_city}_"
        f"{start_date}_"
        f"{end_date}"
    )


def raw_to_bronze(
    city_info,
    file_path,
    start_date,
    end_date
):

    batch_id = make_batch_id(
        city_info,
        start_date,
        end_date
    )

    raw_df = (
        spark.read
        .schema(raw_weather_schema)
        .option("multiLine", "true")
        .json(file_path)
    )

    bronze_df = (
        raw_df
        .withColumn(
            "city",
            lit(city_info["city"])
        )

        .withColumn(
            "country",
            lit(city_info["country"])
        )

        .withColumn(
            "region",
            lit(city_info["region"])
        )

        .withColumn(
            "source",
            lit("Open-Meteo ERA5")
        )

        .withColumn(
            "batch_id",
            lit(batch_id)
        )

        .withColumn(
            "load_type",
            lit(LOAD_TYPE)
        )

        .withColumn(
            "raw_file_path",
            lit(file_path)
        )

        .withColumn(
            "load_timestamp",
            current_timestamp()
        )
    )

    return bronze_df


def bronze_to_silver(bronze_df):

    exploded_df = bronze_df.select(

        "city",
        "country",
        "region",
        "source",
        "batch_id",
        "load_type",
        "raw_file_path",

        col("load_timestamp")
        .alias("bronze_load_timestamp"),

        explode(
            arrays_zip(

                col("hourly.time"),

                col("hourly.temperature_2m"),

                col("hourly.relative_humidity_2m"),

                col("hourly.apparent_temperature"),

                col("hourly.precipitation"),

                col("hourly.rain"),

                col("hourly.snowfall"),

                col("hourly.weather_code"),

                col("hourly.pressure_msl"),

                col("hourly.surface_pressure"),

                col("hourly.cloud_cover"),

                col("hourly.wind_speed_10m"),

                col("hourly.wind_direction_10m"),
            )
        ).alias("weather")
    )

    silver_df = exploded_df.select(

        "city",
        "country",
        "region",

        to_timestamp(
            col("weather.time"),
            "yyyy-MM-dd'T'HH:mm"
        ).alias("timestamp"),

        col("weather.temperature_2m")
        .cast("double")
        .alias("temperature_2m"),

        col("weather.relative_humidity_2m")
        .cast("double")
        .alias("relative_humidity_2m"),

        col("weather.apparent_temperature")
        .cast("double")
        .alias("apparent_temperature"),

        col("weather.precipitation")
        .cast("double")
        .alias("precipitation"),

        col("weather.rain")
        .cast("double")
        .alias("rain"),

        col("weather.snowfall")
        .cast("double")
        .alias("snowfall"),

        col("weather.weather_code")
        .cast("int")
        .alias("weather_code"),

        col("weather.pressure_msl")
        .cast("double")
        .alias("pressure_msl"),

        col("weather.surface_pressure")
        .cast("double")
        .alias("surface_pressure"),

        col("weather.cloud_cover")
        .cast("double")
        .alias("cloud_cover"),

        col("weather.wind_speed_10m")
        .cast("double")
        .alias("wind_speed_10m"),

        col("weather.wind_direction_10m")
        .cast("double")
        .alias("wind_direction_10m"),

        "source",
        "batch_id",
        "load_type",
        "raw_file_path",
        "bronze_load_timestamp",

        current_timestamp()
        .alias("load_timestamp")
    )

    return silver_df


print("Explicit schema and transformations ready.")

Explicit schema and transformations ready.


In [0]:
EXPECTED_TOP_LEVEL_KEYS = {
    "latitude",
    "longitude",
    "generationtime_ms",
    "utc_offset_seconds",
    "timezone",
    "timezone_abbreviation",
    "elevation",
    "hourly_units",
    "hourly"
}

EXPECTED_HOURLY_KEYS = (
    set(WEATHER_COLUMNS)
    | {"time"}
)


def build_raw_file_path(
    city_info,
    start_date,
    end_date
):

    safe_city = (
        city_info["city"]
        .lower()
        .replace(" ", "_")
    )

    return (
        f"{RAW_INCREMENTAL_PATH}/"
        f"{safe_city}_"
        f"{start_date}_"
        f"{end_date}.json"
    )


def raw_file_exists(file_path):

    try:

        dbutils.fs.head(file_path, 1)

        return True

    except Exception:

        return False


def read_raw_json(file_path):

    text = dbutils.fs.head(
        file_path,
        100_000_000
    )

    return json.loads(text)


def validate_raw_file(file_path):

    reasons = []

    try:

        data = read_raw_json(file_path)

    except Exception as e:

        return False, [
            f"Invalid JSON: {e}"
        ]

    missing_top = (
        EXPECTED_TOP_LEVEL_KEYS
        - set(data.keys())
    )

    if missing_top:

        reasons.append(
            "Missing top-level keys: "
            + ", ".join(
                sorted(missing_top)
            )
        )

    hourly = data.get("hourly")

    if not isinstance(hourly, dict):

        return False, (
            reasons
            + ["hourly is not a valid object"]
        )

    missing_hourly = (
        EXPECTED_HOURLY_KEYS
        - set(hourly.keys())
    )

    if missing_hourly:

        reasons.append(
            "Missing hourly keys: "
            + ", ".join(
                sorted(missing_hourly)
            )
        )

    if isinstance(
        hourly.get("time"),
        list
    ):

        expected_length = len(
            hourly["time"]
        )

        if expected_length == 0:

            reasons.append(
                "hourly.time is empty"
            )

        for column in EXPECTED_HOURLY_KEYS:

            if column in hourly:

                if not isinstance(
                    hourly[column],
                    list
                ):

                    reasons.append(
                        f"hourly.{column} "
                        "is not an array"
                    )

                elif (
                    len(hourly[column])
                    != expected_length
                ):

                    reasons.append(
                        f"hourly.{column} "
                        "array length mismatch"
                    )

    else:

        reasons.append(
            "hourly.time is not an array"
        )

    return (
        len(reasons) == 0,
        reasons
    )


def fetch_weather(
    city_info,
    start_date,
    end_date,
    max_retries=MAX_RETRIES
):

    params = {

        "latitude":
            city_info["latitude"],

        "longitude":
            city_info["longitude"],

        "start_date":
            start_date,

        "end_date":
            end_date,

        "hourly":
            ",".join(
                WEATHER_COLUMNS
            ),

        "timezone":
            "UTC",

        "models":
            "era5",
    }

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = requests.get(
                BASE_URL,
                params=params,
                timeout=180
            )

           

            if response.status_code == 429:

                retry_after = (
                    response.headers
                    .get("Retry-After")
                )

                if (
                    retry_after
                    and retry_after.isdigit()
                ):

                    wait_seconds = int(
                        retry_after
                    )

                else:

                    wait_seconds = min(
                        30
                        * (
                            2
                            ** (attempt - 1)
                        ),
                        300
                    )

                print(
                    "429 Too Many Requests →",
                    f"retry {attempt}/"
                    f"{max_retries}",
                    f"in {wait_seconds}s"
                )

                time.sleep(
                    wait_seconds
                )

                continue

           

            if response.status_code in (
                500,
                502,
                503,
                504
            ):

                wait_seconds = min(
                    10 * attempt,
                    60
                )

                print(
                    "Server error",
                    response.status_code,
                    "→ retry in",
                    wait_seconds,
                    "seconds"
                )

                time.sleep(
                    wait_seconds
                )

                continue

            response.raise_for_status()

        
            response.json()

            return response

        except (
            requests.RequestException,
            ValueError
        ) as e:

            if attempt == max_retries:

                raise

            wait_seconds = min(
                10 * attempt,
                60
            )

            print(
                "Request/JSON error →",
                f"retry {attempt}/"
                f"{max_retries}",
                f"in {wait_seconds}s"
            )

            time.sleep(
                wait_seconds
            )

    raise RuntimeError(
        "API request failed after all retries."
    )


def save_raw_response(
    response,
    city_info,
    start_date,
    end_date
):

    file_path = build_raw_file_path(
        city_info,
        start_date,
        end_date
    )

    dbutils.fs.mkdirs(
        RAW_INCREMENTAL_PATH
    )

    dbutils.fs.put(
        file_path,
        response.text,
        overwrite=True
    )

    return file_path


def get_incremental_raw_file(
    city_info,
    start_date,
    end_date
):

    file_path = build_raw_file_path(
        city_info,
        start_date,
        end_date
    )

    
    if FORCE_REFRESH_RAW:

        print(
            "RAW      → FRESH API REQUEST"
        )

        response = fetch_weather(
            city_info,
            start_date,
            end_date
        )

        file_path = save_raw_response(
            response,
            city_info,
            start_date,
            end_date
        )

        valid, reasons = validate_raw_file(
            file_path
        )

        if not valid:

            raise ValueError(
                "Downloaded raw file failed validation: "
                + " | ".join(reasons)
            )

        print(
            "RAW      → DOWNLOADED",
            f"{len(response.content) / (1024 ** 2):.3f} MB"
        )

        return file_path

  
    if raw_file_exists(file_path):

        valid, reasons = validate_raw_file(
            file_path
        )

        if valid:

            print(
                "RAW      → VALID EXISTING FILE REUSED"
            )

            return file_path

    response = fetch_weather(
        city_info,
        start_date,
        end_date
    )

    file_path = save_raw_response(
        response,
        city_info,
        start_date,
        end_date
    )

    return file_path


def write_execution_log(
    run_id,
    batch_id,
    layer,
    parameter_or_file,
    start_time,
    end_time,
    status,
    rows_inserted=0,
    rows_updated=0,
    error_message=None
):

    record = [{

        "run_id":
            run_id,

        "batch_id":
            batch_id,

        "layer":
            layer,

        "load_type":
            LOAD_TYPE,

        "parameter_or_file":
            parameter_or_file,

        "start_time":
            start_time,

        "end_time":
            end_time,

        "status":
            status,

        "rows_inserted":
            int(rows_inserted),

        "rows_updated":
            int(rows_updated),

        "error_message":
            error_message
    }]

    (
        spark.createDataFrame(
            record,
            schema=(
                spark
                .table(EXECUTION_LOG_TABLE)
                .schema
            )
        )

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(
            EXECUTION_LOG_TABLE
        )
    )


def write_error_log(
    run_id,
    batch_id,
    layer,
    source_file,
    error
):

    record = [{

        "error_id":
            str(uuid.uuid4()),

        "run_id":
            run_id,

        "batch_id":
            batch_id,

        "layer":
            layer,

        "source_file":
            source_file,

        "error_type":
            type(error).__name__,

        "error_message":
            str(error),

        "error_timestamp":
            datetime.now()
    }]

    (
        spark.createDataFrame(
            record,
            schema=(
                spark
                .table(ERROR_LOG_TABLE)
                .schema
            )
        )

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(
            ERROR_LOG_TABLE
        )
    )


print("API, validation and logging helpers ready.")

API, validation and logging helpers ready.


In [0]:
def merge_bronze(bronze_df):

    target = DeltaTable.forName(
        spark,
        BRONZE_TABLE
    )

    (
        target
        .alias("target")

        .merge(
            bronze_df.alias("source"),
            """
            target.batch_id =
            source.batch_id
            """
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

    metrics = (
        target
        .history(1)
        .select(
            "operationMetrics"
        )
        .first()[
            "operationMetrics"
        ]
    )

    inserted = int(
        metrics.get(
            "numTargetRowsInserted",
            0
        )
    )

    updated = int(
        metrics.get(
            "numTargetRowsUpdated",
            0
        )
    )

    return inserted, updated


def merge_silver(silver_df):

 
    staged_df = (
        silver_df
        .dropDuplicates(
            ["city", "timestamp"]
        )
    )

    target = DeltaTable.forName(
        spark,
        SILVER_TABLE
    )

  
    change_condition = (
        "NOT ("
        + " AND ".join(
            [
                f"target.{column} "
                f"<=> "
                f"source.{column}"

                for column
                in WEATHER_COLUMNS
            ]
        )
        + ")"
    )

    update_values = {

        column:
            f"source.{column}"

        for column
        in WEATHER_COLUMNS
    }

    update_values.update({

        "country":
            "source.country",

        "region":
            "source.region",

        "source":
            "source.source",

        "batch_id":
            "source.batch_id",

        "load_type":
            "source.load_type",

        "raw_file_path":
            "source.raw_file_path",

        "bronze_load_timestamp":
            "source.bronze_load_timestamp",

        "load_timestamp":
            "current_timestamp()"
    })

    (
        target
        .alias("target")

        .merge(
            staged_df.alias("source"),

            """
            target.city = source.city
            AND
            target.timestamp = source.timestamp
            """
        )

        .whenMatchedUpdate(
            condition=change_condition,
            set=update_values
        )

        .whenNotMatchedInsertAll()

        .execute()
    )

    metrics = (
        target
        .history(1)
        .select(
            "operationMetrics"
        )
        .first()[
            "operationMetrics"
        ]
    )

    inserted = int(
        metrics.get(
            "numTargetRowsInserted",
            0
        )
    )

    updated = int(
        metrics.get(
            "numTargetRowsUpdated",
            0
        )
    )

    return inserted, updated


print("Bronze and Silver MERGE functions ready.")

Bronze and Silver MERGE functions ready.


In [0]:
def run_incremental_batch(
    city_info,
    start_date,
    end_date
):

    run_id = str(
        uuid.uuid4()
    )

    batch_id = make_batch_id(
        city_info,
        start_date,
        end_date
    )

    pipeline_start = datetime.now()

    file_path = build_raw_file_path(
        city_info,
        start_date,
        end_date
    )

    print(
        "\n======================================"
    )

    print(
        city_info["city"],
        start_date,
        "→",
        end_date
    )

    print(
        "Batch:",
        batch_id
    )

    try:

     

        raw_start = datetime.now()

        file_path = get_incremental_raw_file(
            city_info,
            start_date,
            end_date
        )

        raw_end = datetime.now()

        write_execution_log(

            run_id=run_id,

            batch_id=batch_id,

            layer="Raw",

            parameter_or_file=file_path,

            start_time=raw_start,

            end_time=raw_end,

            status="SUCCESS"
        )

      

        bronze_start = datetime.now()

        bronze_df = raw_to_bronze(
            city_info,
            file_path,
            start_date,
            end_date
        )

        (
            bronze_inserted,
            bronze_updated
        ) = merge_bronze(
            bronze_df
        )

        bronze_end = datetime.now()

        write_execution_log(

            run_id=run_id,

            batch_id=batch_id,

            layer="Bronze",

            parameter_or_file=file_path,

            start_time=bronze_start,

            end_time=bronze_end,

            status="SUCCESS",

            rows_inserted=
                bronze_inserted,

            rows_updated=
                bronze_updated
        )

        print(
            "BRONZE   → SUCCESS",
            "| inserted:",
            bronze_inserted,
            "| updated:",
            bronze_updated
        )

      

        silver_start = datetime.now()

        silver_stage_df = (
            bronze_to_silver(
                bronze_df
            )

            .dropDuplicates(
                ["city", "timestamp"]
            )
        )

        null_timestamps = (
            silver_stage_df
            .filter(
                col("timestamp")
                .isNull()
            )
            .count()
        )

        if null_timestamps > 0:

            raise ValueError(
                "Silver staging contains "
                f"{null_timestamps} "
                "null timestamp(s)"
            )

     

        (
            silver_inserted,
            silver_updated
        ) = merge_silver(
            silver_stage_df
        )

        silver_end = datetime.now()

        write_execution_log(

            run_id=run_id,

            batch_id=batch_id,

            layer="Silver",

            parameter_or_file=file_path,

            start_time=silver_start,

            end_time=silver_end,

            status="SUCCESS",

            rows_inserted=
                silver_inserted,

            rows_updated=
                silver_updated
        )

        print(
            "SILVER   → SUCCESS",
            "| inserted:",
            silver_inserted,
            "| updated:",
            silver_updated
        )

        print(
            "FINAL    → SUCCESS"
        )

        return {

            "status":
                "SUCCESS",

            "batch_id":
                batch_id,

            "silver_inserted":
                silver_inserted,

            "silver_updated":
                silver_updated,

            "bronze_inserted":
                bronze_inserted,

            "bronze_updated":
                bronze_updated
        }

    except Exception as e:

        failure_time = datetime.now()

        print(
            "FINAL    → FAILURE"
        )

        print(
            "Reason:",
            str(e)
        )

        write_execution_log(

            run_id=run_id,

            batch_id=batch_id,

            layer="Pipeline",

            parameter_or_file=
                file_path,

            start_time=
                pipeline_start,

            end_time=
                failure_time,

            status=
                "FAILURE",

            rows_inserted=0,

            rows_updated=0,

            error_message=
                str(e)
        )

        write_error_log(

            run_id=run_id,

            batch_id=batch_id,

            layer="Pipeline",

            source_file=file_path,

            error=e
        )

        return {

            "status":
                "FAILURE",

            "batch_id":
                batch_id,

            "error":
                str(e)
        }


print("Incremental batch runner ready.")

Incremental batch runner ready.


In [0]:
print(
    "===== INCREMENTAL PRE-FLIGHT ====="
)

print(
    "Target years:",
    INCREMENTAL_START_YEAR,
    "→",
    INCREMENTAL_END_YEAR
)

print(
    "Cities:",
    len(CITIES)
)

print(
    "Expected batches:",
    TOTAL_BATCHES
)

print(
    "Load type:",
    LOAD_TYPE
)

print(
    "Raw path:",
    RAW_INCREMENTAL_PATH
)

print(
    "Fresh API fetching:",
    FORCE_REFRESH_RAW
)

print(
    "Current Silver rows:",
    spark.table(
        SILVER_TABLE
    ).count()
)

date_range = (
    spark.table(SILVER_TABLE)
    .selectExpr(
        "MIN(timestamp) AS min_timestamp",
        "MAX(timestamp) AS max_timestamp"
    )
    .first()
)

print(
    "Current Silver earliest:",
    date_range["min_timestamp"]
)

print(
    "Current Silver latest:",
    date_range["max_timestamp"]
)

===== INCREMENTAL PRE-FLIGHT =====
Target years: 2021 → 2025
Cities: 9
Expected batches: 45
Load type: INCREMENTAL
Raw path: /Volumes/workspace/default/products/european_climate/raw/incremental
Fresh API fetching: True
Current Silver rows: 3155760
Current Silver earliest: 1981-01-01 00:00:00
Current Silver latest: 2020-12-31 23:00:00


In [0]:
incremental_success = 0
incremental_failed = 0

total_inserted = 0
total_updated = 0

failed_batches = []

batch_number = 0


print(
    "======================================"
)

print(
    "INCREMENTAL LOAD"
)

print(
    "Years:",
    INCREMENTAL_START_YEAR,
    "→",
    INCREMENTAL_END_YEAR
)

print(
    "Expected batches:",
    TOTAL_BATCHES
)

print(
    "======================================"
)


for city_info in CITIES:

    for year in range(
        INCREMENTAL_START_YEAR,
        INCREMENTAL_END_YEAR + 1
    ):

        batch_number += 1

        start_date = (
            f"{year}-01-01"
        )

        end_date = (
            f"{year}-12-31"
        )

        print(
            f"\nBatch "
            f"{batch_number}/"
            f"{TOTAL_BATCHES}"
        )

        result = run_incremental_batch(

            city_info=
                city_info,

            start_date=
                start_date,

            end_date=
                end_date
        )

        if (
            result["status"]
            == "SUCCESS"
        ):

            incremental_success += 1

            total_inserted += (
                result[
                    "silver_inserted"
                ]
            )

            total_updated += (
                result[
                    "silver_updated"
                ]
            )

        else:

            incremental_failed += 1

            failed_batches.append({

                "city":
                    city_info["city"],

                "year":
                    year,

                "batch_id":
                    result["batch_id"],

                "error":
                    result.get(
                        "error",
                        "Unknown error"
                    )
            })

        print(
            "Progress:",
            batch_number,
            "/",
            TOTAL_BATCHES,
            "| Success:",
            incremental_success,
            "| Failed:",
            incremental_failed
        )

      
        time.sleep(
            NORMAL_DELAY_SECONDS
        )

      
        if (
            batch_number % 20
            == 0
        ):

            print(
                "Cooling down API "
                "for 60 seconds..."
            )

            time.sleep(60)


print(
    "\n======================================"
)

print(
    "INCREMENTAL PASS FINISHED"
)

print(
    "Success:",
    incremental_success
)

print(
    "Failed:",
    incremental_failed
)

print(
    "Silver rows inserted:",
    total_inserted
)

print(
    "Silver rows updated:",
    total_updated
)

print(
    "======================================"
)


if failed_batches:

    print(
        "\n===== FAILED BATCHES ====="
    )

    for item in failed_batches:

        print(
            item["city"],
            item["year"],
            "→",
            item["error"]
        )

else:

    print(
        "\nNo failed batches."
    )

INCREMENTAL LOAD
Years: 2021 → 2025
Expected batches: 45

Batch 1/45

Oslo 2021-01-01 → 2021-12-31
Batch: incremental_oslo_2021-01-01_2021-12-31
RAW      → FRESH API REQUEST
Wrote 645670 bytes.
RAW      → DOWNLOADED 0.616 MB
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 1 / 45 | Success: 1 | Failed: 0

Batch 2/45

Oslo 2022-01-01 → 2022-12-31
Batch: incremental_oslo_2022-01-01_2022-12-31
RAW      → FRESH API REQUEST
Wrote 644737 bytes.
RAW      → DOWNLOADED 0.615 MB
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 2 / 45 | Success: 2 | Failed: 0

Batch 3/45

Oslo 2023-01-01 → 2023-12-31
Batch: incremental_oslo_2023-01-01_2023-12-31
RAW      → FRESH API REQUEST
Wrote 644640 bytes.
RAW      → DOWNLOADED 0.615 MB
BRONZE   → SUCCESS | inserted: 1 | updated: 0
SILVER   → SUCCESS | inserted: 8760 | updated: 0
FINAL    → SUCCESS
Progress: 3 /

In [0]:
from pyspark.sql.functions import (
    min,
    max
)




try:

    raw_files = dbutils.fs.ls(
        RAW_INCREMENTAL_PATH
    )

    json_files = [
        f
        for f in raw_files
        if f.name.endswith(".json")
    ]

    raw_bytes = sum(
        f.size
        for f in json_files
    )

    raw_mb = (
        raw_bytes
        / (1024 ** 2)
    )

except Exception:

    json_files = []
    raw_mb = 0




silver_df = spark.table(
    SILVER_TABLE
)

silver_rows = (
    silver_df.count()
)

distinct_keys = (
    silver_df
    .select(
        "city",
        "timestamp"
    )
    .distinct()
    .count()
)

duplicates = (
    silver_rows
    - distinct_keys
)

null_timestamps = (
    silver_df
    .filter(
        col("timestamp")
        .isNull()
    )
    .count()
)

null_load_timestamp = (
    silver_df
    .filter(
        col("load_timestamp")
        .isNull()
    )
    .count()
)

date_range = (
    silver_df
    .selectExpr(

        "MIN(timestamp) "
        "AS min_timestamp",

        "MAX(timestamp) "
        "AS max_timestamp"
    )
    .first()
)




incremental_logs = (

    spark
    .table(
        EXECUTION_LOG_TABLE
    )

    .filter(
        col("load_type")
        == "INCREMENTAL"
    )

    .filter(
        col("layer")
        == "Silver"
    )

    .filter(
        col("status")
        == "SUCCESS"
    )
)


print(
    "===== INCREMENTAL VALIDATION ====="
)

print(
    "Incremental raw JSON files:",
    len(json_files)
)

print(
    "Incremental raw size:",
    round(raw_mb, 2),
    "MB"
)

print(
    "Current Silver rows:",
    silver_rows
)

print(
    "Distinct city + timestamp:",
    distinct_keys
)

print(
    "Duplicates:",
    duplicates
)

print(
    "Null timestamps:",
    null_timestamps
)

print(
    "Null load_timestamp:",
    null_load_timestamp
)

print(
    "Earliest timestamp:",
    date_range["min_timestamp"]
)

print(
    "Latest timestamp:",
    date_range["max_timestamp"]
)

print(
    "Current run inserted:",
    total_inserted
)

print(
    "Current run updated:",
    total_updated
)


print(
    "\n===== CHECKLIST ====="
)

checks = {

    "No duplicate city + timestamp":
        duplicates == 0,

    "No null timestamps":
        null_timestamps == 0,

    "No null load timestamps":
        null_load_timestamp == 0,

    "No failed batches":
        incremental_failed == 0,

    "All chunk batches successful":
        incremental_success
        == TOTAL_BATCHES
}


for name, passed in checks.items():

    print(
        name,
        "→",
        "PASS"
        if passed
        else "FAIL"
    )


if all(
    checks.values()
):

    print(
        "\nINCREMENTAL CHUNK "
        "VALIDATION PASSED"
    )

else:

    print(
        "\nCHECK FAILED ITEMS"
    )

===== INCREMENTAL VALIDATION =====
Incremental raw JSON files: 130
Incremental raw size: 57.52 MB
Current Silver rows: 3550176
Distinct city + timestamp: 3550176
Duplicates: 0
Null timestamps: 0
Null load_timestamp: 0
Earliest timestamp: 1981-01-01 00:00:00
Latest timestamp: 2025-12-31 23:00:00
Current run inserted: 394416
Current run updated: 0

===== CHECKLIST =====
No duplicate city + timestamp → PASS
No null timestamps → PASS
No null load timestamps → PASS
No failed batches → PASS
All chunk batches successful → PASS

INCREMENTAL CHUNK VALIDATION PASSED


In [0]:
from pyspark.sql.functions import col

logs = spark.table("pipeline_execution_logs")

incremental_silver = logs.filter(
    (col("load_type") == "INCREMENTAL") &
    (col("layer") == "Silver") &
    (col("status") == "SUCCESS")
)

display(
    incremental_silver.select(
        "batch_id",
        "rows_inserted",
        "rows_updated"
    ).orderBy("batch_id")
)

print(
    "Successful Silver batch logs:",
    incremental_silver.select("batch_id").distinct().count()
)

batch_id,rows_inserted,rows_updated
incremental_berlin_2016-01-01_2016-12-31,0,0
incremental_berlin_2017-01-01_2017-12-31,0,0
incremental_berlin_2018-01-01_2018-12-31,0,0
incremental_berlin_2019-01-01_2019-12-31,0,0
incremental_berlin_2020-01-01_2020-12-31,0,0
incremental_berlin_2021-01-01_2021-12-31,8760,0
incremental_berlin_2022-01-01_2022-12-31,8760,0
incremental_berlin_2023-01-01_2023-12-31,8760,0
incremental_berlin_2024-01-01_2024-12-31,8784,0
incremental_berlin_2025-01-01_2025-12-31,8760,0


Successful Silver batch logs: 90


In [0]:
from pyspark.sql.functions import (
    col, countDistinct, min, max
)

silver = spark.table("silver_weather")

print("===== FINAL PRODUCTION VALIDATION =====")

total_rows = silver.count()

distinct_keys = (
    silver.select("city", "timestamp")
    .distinct()
    .count()
)

null_timestamps = silver.filter(
    col("timestamp").isNull()
).count()

null_load_timestamps = silver.filter(
    col("load_timestamp").isNull()
).count()

print("Total rows:", total_rows)
print("Duplicate keys:", total_rows - distinct_keys)
print("Null timestamps:", null_timestamps)
print("Null load timestamps:", null_load_timestamps)

silver.agg(
    countDistinct("city").alias("total_cities"),
    min("timestamp").alias("start_date"),
    max("timestamp").alias("end_date")
).show()

display(
    silver.groupBy("city")
    .count()
    .orderBy("city")
)

print("===== SILVER SCHEMA =====")
silver.printSchema()

print("===== SAMPLE RECORDS =====")
display(silver.limit(20))

assert total_rows == 3550176
assert total_rows == distinct_keys
assert null_timestamps == 0
assert null_load_timestamps == 0

print("FINAL PRODUCTION VALIDATION PASSED")

===== FINAL PRODUCTION VALIDATION =====
Total rows: 3550176
Duplicate keys: 0
Null timestamps: 0
Null load timestamps: 0
+------------+-------------------+-------------------+
|total_cities|         start_date|           end_date|
+------------+-------------------+-------------------+
|           9|1981-01-01 00:00:00|2025-12-31 23:00:00|
+------------+-------------------+-------------------+



city,count
Berlin,394464
Copenhagen,394464
Helsinki,394464
London,394464
Madrid,394464
Oslo,394464
Rome,394464
Stockholm,394464
Warsaw,394464


===== SILVER SCHEMA =====
root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- source: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- raw_file_path: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = true)
 |-- load_timestamp: timestamp (nullable = tr

city,country,region,timestamp,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,source,batch_id,load_type,raw_file_path,bronze_load_timestamp,load_timestamp
Warsaw,Poland,Eastern Europe,2025-01-01T04:00:00.000Z,1.0,80.0,-5.0,0.0,0.0,0.0,3,1017.3,1003.1,99.0,23.9,241.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-01T16:00:00.000Z,4.8,68.0,-1.3,0.0,0.0,0.0,3,1010.8,996.9,100.0,26.8,231.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-02T01:00:00.000Z,6.9,45.0,-0.2,0.0,0.0,0.0,3,1004.0,990.3,99.0,30.6,236.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-03T13:00:00.000Z,1.1,85.0,-4.1,0.1,0.1,0.0,51,1006.4,992.4,96.0,20.0,246.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-03T14:00:00.000Z,1.0,84.0,-4.4,0.1,0.0,0.07,71,1006.2,992.1,87.0,20.9,250.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-04T18:00:00.000Z,-5.3,81.0,-10.0,0.0,0.0,0.0,3,1019.4,1004.8,81.0,10.8,234.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-05T02:00:00.000Z,-4.4,86.0,-9.3,0.0,0.0,0.0,2,1016.4,1001.9,77.0,13.8,208.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-05T11:00:00.000Z,-1.7,73.0,-6.2,0.0,0.0,0.0,3,1013.5,999.2,99.0,10.5,200.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-06T02:00:00.000Z,-2.5,91.0,-7.6,0.3,0.0,0.21,73,1002.6,988.4,100.0,16.8,139.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z
Warsaw,Poland,Eastern Europe,2025-01-06T17:00:00.000Z,4.4,87.0,-0.1,0.0,0.0,0.0,3,999.2,985.4,99.0,18.7,203.0,Open-Meteo ERA5,incremental_warsaw_2025-01-01_2025-12-31,INCREMENTAL,/Volumes/workspace/default/products/european_climate/raw/incremental/warsaw_2025-01-01_2025-12-31.json,2026-10-06T20:55:35.054Z,2026-10-06T20:55:35.054Z


FINAL PRODUCTION VALIDATION PASSED


In [0]:
from pyspark.sql.functions import col

logs = spark.table("pipeline_execution_logs")

incremental_silver = logs.filter(
    (col("load_type") == "INCREMENTAL") &
    (col("layer") == "Silver") &
    (col("status") == "SUCCESS")
)

batch_count = (
    incremental_silver
    .select("batch_id")
    .distinct()
    .count()
)

print("Successful incremental Silver batches:", batch_count)

display(
    incremental_silver
    .select(
        "batch_id",
        "rows_inserted",
        "rows_updated",
        "status"
    )
    .orderBy("batch_id")
)

assert batch_count == 90

print("INCREMENTAL LOG VALIDATION PASSED")

Successful incremental Silver batches: 90


batch_id,rows_inserted,rows_updated,status
incremental_berlin_2016-01-01_2016-12-31,0,0,SUCCESS
incremental_berlin_2017-01-01_2017-12-31,0,0,SUCCESS
incremental_berlin_2018-01-01_2018-12-31,0,0,SUCCESS
incremental_berlin_2019-01-01_2019-12-31,0,0,SUCCESS
incremental_berlin_2020-01-01_2020-12-31,0,0,SUCCESS
incremental_berlin_2021-01-01_2021-12-31,8760,0,SUCCESS
incremental_berlin_2022-01-01_2022-12-31,8760,0,SUCCESS
incremental_berlin_2023-01-01_2023-12-31,8760,0,SUCCESS
incremental_berlin_2024-01-01_2024-12-31,8784,0,SUCCESS
incremental_berlin_2025-01-01_2025-12-31,8760,0,SUCCESS


INCREMENTAL LOG VALIDATION PASSED
